# 99 — Combine Example Factors

Join the three example factors into a single wide feature frame aligned by (timestamp, symbol) and persist it to the processed parquet store for the trainer.

> **Outputs are not committed.** These notebooks read the shared parquet store (`../../data/...`) populated by `prodigy-data backfill`. Run `jupyter nbconvert --execute --inplace research/notebooks/*.ipynb` (or Run All in Jupyter) after a backfill to populate and save outputs. Code paths are verified by `tests/test_research_notebooks.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from prodigy.data.parquet_store import load_funding_rates, load_ohlcv
from prodigy.factors.examples import (
    example_funding_factor,
    example_momentum_factor,
    example_volatility_factor,
)
from prodigy.research.backtester import Backtester
from prodigy.research.signals import SignalParams, score_to_lot_signals


In [ ]:
# Research params. DATA_ROOT is two levels up from research/notebooks/.
DATA_ROOT = "../../data"
SYMBOL = "ETH/USDT:USDT"
START = "2024-01-01"
END = "2024-04-01"
TIMEFRAME = "15m"


In [ ]:
ohlcv = load_ohlcv(DATA_ROOT, SYMBOL, START, END, TIMEFRAME)
funding = load_funding_rates(DATA_ROOT, SYMBOL, START, END)
ohlcv.shape, funding.shape


In [ ]:
momentum = example_momentum_factor(ohlcv, lookback_bars=4)
ffactor = example_funding_factor(funding, window=20)
volatility = example_volatility_factor(ohlcv, atr_window=14)


In [ ]:
# Pivot each factor frame to one column, then join on (timestamp, symbol).
def to_column(factor, name):
    return factor.rename(columns={"value": name})[["timestamp", "symbol", name]]

features = (
    ohlcv[["timestamp", "symbol", "open", "high", "low", "close", "volume"]]
    .merge(to_column(momentum, "example_momentum"), on=["timestamp", "symbol"], how="left")
    .merge(to_column(ffactor, "example_funding"), on=["timestamp", "symbol"], how="left")
    .merge(to_column(volatility, "example_volatility"), on=["timestamp", "symbol"], how="left")
)
features.head()


In [ ]:
features.to_parquet(
    "../../data/processed/example_features.parquet.gzip",
    compression="gzip",
    index=False,
)
features.shape


In [ ]:
# Smoke-test the saved feature frame through the Backtester using the
# combined momentum column as the factor value.
combined_factor = features[["timestamp", "symbol"]].assign(
    factor_name="example_momentum",
    value=features["example_momentum"],
)
bt = Backtester(prices=ohlcv, factors=combined_factor, funding=funding)
bt.run_full_report()
